In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta
import numpy as np


In [ ]:
# 신체 계측 정보 경로
table_dir = str(PROCESSED_DIR / 'AEGIS-Weight-and-Height/de')

# 환자별 노이즈 정보
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
patient_mapping_df['new_pt_id'] = patient_mapping_df['new_pt_id'].astype(str)

patient_id_map = patient_mapping_df.set_index('pt_id')['new_pt_id'].to_dict()
date_offset_map = patient_mapping_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_offset_map = patient_mapping_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(table_dir, '**/*.csv'), recursive = True)
print(f"[INFO] 총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# 컬럼 삭제
print("\n[INFO] 컬럼 제거")

columns_to_drop = [
    'prone_bone', 'chest', 'head', 'sit_height'
]

for file in csv_files:
    try:
        body_measurements_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        total_cols_to_remove = len(columns_to_drop)
        present_columns = [col for col in columns_to_drop if col in body_measurements_df.columns]
        removed_cols_count = len(present_columns)

        if removed_cols_count > 0:
            body_measurements_df.drop(columns = present_columns, inplace = True)

        body_measurements_df.to_csv(file, index=False, encoding = 'utf-8')

        print(f"[INFO] {file} - 제거 대상: {total_cols_to_remove}개, 실제 제거: {removed_cols_count}개")

    except Exception as e:
        print(f"[ERROR] 오류: {file} → {e}")


In [ ]:
# pt_id 치환
for file in csv_files:
    try:
        body_measurements_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        if 'pt_id' in body_measurements_df.columns:
            body_measurements_df['pt_id'] = body_measurements_df['pt_id'].map(patient_id_map).fillna(body_measurements_df['pt_id'])


        body_measurements_df.to_csv(file, index = False, encoding = 'utf-8')
        print(f"[INFO] pt_id 변경 완료: {file}")

    except Exception as e:
        print(f"[ERROR] pt_id 오류: {file} -> {e}")


In [ ]:
# hos_dt_first 처리
for file in csv_files:
    try:
        body_measurements_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        if 'hos_dt_first' in body_measurements_df.columns:
            body_measurements_df = body_measurements_df.rename(columns = {'hos_dt_first': 'host_dt_first'})

        if 'host_dt_first' in body_measurements_df.columns:
            body_measurements_df['host_dt_first'] = pd.to_datetime(body_measurements_df['host_dt_first'], errors = 'coerce')

            body_measurements_df['host_dt_first'] = body_measurements_df['host_dt_first'] \
                + pd.to_timedelta(body_measurements_df['pt_id'].map(date_offset_map).fillna(0).astype(int), unit = "D") \
                + pd.to_timedelta(body_measurements_df['pt_id'].map(time_offset_map).fillna(0).astype(int), unit = "m")

            print(body_measurements_df.head(3))

            body_measurements_df.to_csv(file, index = False, encoding = 'utf-8')
            print(f"[INFO] host_dt_first 완료: {file}")


    except Exception as e:
        print(f"[ERROR] host_dt_first 오류: {file} -> {e}")


In [ ]:
# tx_date 처리
for file in csv_files:
    try:
        body_measurements_df = pd.read_csv(file, dtype=str, sep=",", quotechar ='"', lineterminator = '\n')

        if 'tx_date' in body_measurements_df.columns:
            body_measurements_df['tx_date'] = pd.to_datetime(body_measurements_df['tx_date'], errors = 'coerce')
            patient_ids = body_measurements_df['pt_id']
            body_measurements_df['tx_date'] = body_measurements_df['tx_date'] + patient_ids.map(lambda x: timedelta(days = date_offset_map.get(x, 0)))

            print(body_measurements_df.head(3))

            body_measurements_df.to_csv(file, index = False, encoding = 'utf-8')
            print(f"[INFO] tx_date 완료: {file}")

    except Exception as e:
        print(f"[ERROR] tx_date 오류: {file} -> {e}")


In [ ]:
# body_ms_dt 처리
for file in csv_files:
    try:
        body_measurements_df = pd.read_csv(file, dtype=str, sep=",", quotechar ='"', lineterminator = '\n')

        if 'body_ms_dt' in body_measurements_df.columns:
            body_measurements_df['body_ms_dt'] = pd.to_datetime(body_measurements_df['body_ms_dt'], errors = 'coerce')

            body_measurements_df['body_ms_dt'] = body_measurements_df['body_ms_dt'] \
                + pd.to_timedelta(body_measurements_df['pt_id'].map(date_offset_map).fillna(0).astype(int), unit = "D") \
                + pd.to_timedelta(body_measurements_df['pt_id'].map(time_offset_map).fillna(0).astype(int), unit = "m")

            print(body_measurements_df.head(3))

            body_measurements_df.to_csv(file, index = False, encoding = 'utf-8')
            print(f"[INFO] body_ms_dt 완료: {file}")

    except Exception as e:
        print(f"[ERROR] body_ms_dt 오류: {file} -> {e}")


In [ ]:
# vs_date 처리
for file in csv_files:
    try:
        body_measurements_df = pd.read_csv(file, dtype=str, sep=",", quotechar ='"', lineterminator = '\n')

        if 'vs_date' in body_measurements_df.columns:
            body_measurements_df['vs_date'] = pd.to_datetime(body_measurements_df['vs_date'], errors = 'coerce')
            patient_ids = body_measurements_df['pt_id']
            body_measurements_df['vs_date'] = body_measurements_df['vs_date'] + patient_ids.map(lambda x: timedelta(days = date_offset_map.get(x, 0)))

            print(body_measurements_df.head(3))

            body_measurements_df.to_csv(file, index = False, encoding = 'utf-8')
            print(f"[INFO] vs_date 완료: {file}")

    except Exception as e:
        print(f"[ERROR] vs_date 오류: {file} -> {e}")


In [ ]:
# weight 범주화
for file in csv_files:
    try:
        body_measurements_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        # weight 처리
        if 'weight' in body_measurements_df.columns:
            body_measurements_df['weight'] = pd.to_numeric(body_measurements_df['weight'], errors = 'coerce').fillna(0)

            body_measurements_df['weight'] = np.floor(body_measurements_df['weight'].astype(float) / 5) * 5

            body_measurements_df.to_csv(file, index = False, encoding = 'utf-8')
            print(body_measurements_df['weight'].head(5))


            print(f"[INFO] 범주화 완료: {file}")
        else:
            print(f"[WARN] 'weight' 컬럼 없음: {file}")

    except Exception as e:
        print(f"[ERROR] weight 오류: {file} -> {e}")


In [ ]:
# height 범주화
for file in csv_files:
    try:
        body_measurements_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        # weight 처리
        if 'height' in body_measurements_df.columns:
            body_measurements_df['height'] = np.floor(body_measurements_df['height'].astype(float) / 5) * 5

            body_measurements_df.to_csv(file, index = False, encoding = 'utf-8')
            print(body_measurements_df['height'].head(5))


            print(f"[INFO] 범주화 완료: {file}")
        else:
            print(f"[WARN] 'height' 컬럼 없음: {file}")

    except Exception as e:
        print(f"[ERROR] height 오류: {file} -> {e}")
